In [14]:
import baltic as bt
import pandas as pd
import numpy as np
import json
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import itertools
from collections import defaultdict
import statistics
import numpy as np
import re
import random
import math
import scipy.stats as stats
from collections import Counter
from scipy.stats import gaussian_kde
import seaborn as sns

In [ ]:
json_path = "clock_rate/host_clock_rates.json"

with open(json_path) as f:
    host_stats = json.load(f)

In [16]:
print(host_stats)

{'na_avian': {'clock_rate': 0.0014153628923017929, 'root_yr': 1932.7972743811047}, 'eurasian_avian': {'clock_rate': 0.0010022429427303807, 'root_yr': 1925.5975498054086}, 'swine': {'clock_rate': 0.003923517661115446, 'root_yr': 1965.3744788462288}, 'canine': {'clock_rate': 0.0021760366926091234, 'root_yr': 2004.8254515694057}, 'human': {'clock_rate': 0.00417372439954042, 'root_yr': 1967.4178682706547}, 'equine': {'clock_rate': 0.001865099192944831, 'root_yr': 1954.4550337364963}}


In [21]:
clades = ['na_avian', 'eurasian_avian', 'swine']

# reassorted lineages

In [18]:
# visit each reassorted node
# find how far it goes until another reassortment event or leaf
# sum all the edges
# keep that in a dictionary

In [19]:
def max_distance(node):
    """
    Return the maximum distance from `node` down to the first downstream
    reassortment OR a leaf, taken over all descendant paths.
    """

    if node.is_leaf() or node.traits.get('is_reassorted'):
        return node.length
    
    else:
        return node.length + max(max_distance(child) for child in node.children)

In [ ]:
clade_persistences = {clade: {} for clade in clades}

for clade in clades:
    
    clock_rate = host_stats[clade]["clock_rate"]
    
    for i in range(1000):
        
        tree = f'hosts/{clade}/trees/randomized_tree_{i}.nwk'
        mytree= bt.loadNewick(tree, absoluteTime= False)
        
        scaled_distances = []
    
        for k in mytree.Objects:

            # skip leaves
            if k.is_leaf():
                continue

            # only look at reassorted nodes
            if k.is_node() and k.traits.get('is_reassorted'):

                children = k.children

                # if both children are reassorted or leaves, take max length, skip recursion
                if all(child.traits.get('is_reassorted') or child.is_leaf() for child in children):

                    max_child_length = max(child.length for child in children)

                else:

                    child_path_lengths = [max_distance(child) for child in children]
                    max_child_length = max(child_path_lengths)
                    
                scaled_distance = max_child_length / clock_rate
                scaled_distances.append(scaled_distance)

        clade_persistences[clade][i] = scaled_distances

In [23]:
nonrea_clade_persistences = {clade: {} for clade in clades}

for clade in clades:
     
    clock_rate = host_stats[clade]["clock_rate"]

    for i in range(1000):

        tree = f'hosts/{clade}/trees/randomized_tree_{i}.nwk'
        mytree= bt.loadNewick(tree, absoluteTime= False)
        
        scaled_distances = []

        for k in mytree.Objects:

            # skip leaves
            if k.is_leaf():
                continue

            # only look at reassorted nodes
            if k.is_node() and not k.traits.get('is_reassorted'):

                children = k.children

                # if both children are reassorted or leaves, take max length, skip recursion
                if all(child.traits.get('is_reassorted') or child.is_leaf() for child in children):

                    max_child_length = max(child.length for child in children)

                else:

                    child_path_lengths = [max_distance(child) for child in children]
                    max_child_length = max(child_path_lengths)

                scaled_distance = max_child_length / clock_rate
                scaled_distances.append(scaled_distance)

        nonrea_clade_persistences[clade][i] = scaled_distances


In [24]:
with open("clade_persistences.json", "w") as f:
    json.dump(clade_persistences, f)
    
with open("nonrea_clade_persistences.json", "w") as f:
    json.dump(nonrea_clade_persistences, f)

# BOTH

In [25]:
with open("nonrea_clade_persistences.json", "r") as f:
    nonrea_clade_persistences = json.load(f)

with open("clade_persistences.json", "r") as f:
    clade_persistences = json.load(f)

In [ ]:
def get_binned_proportions_replicates(all_replicates):
    
    """
    all_replicates: list of lists of persistence times for each replicate
    returns: bins (only bins that appear), mean_proportions, lower_ci, upper_ci
    """

    replicate_props = []
    replicate_bins = []

    for rep in all_replicates:
        rep = np.array(rep)

        binned = np.floor(rep).astype(int)
        counts = Counter(binned)
        total = len(binned)

        bins = np.array(sorted(counts.keys()))
        proportions = np.array([counts[b]/total for b in bins])

        replicate_props.append(proportions)
        replicate_bins.append(bins)

    all_bins = sorted(set(np.concatenate(replicate_bins))) # union of all bins
    mean_props = []
    lower_ci = []
    upper_ci = []

    for b in all_bins:
        # collect proportion for each bin across replicates (0 if replicate didn't have it)
        props = np.array([rep_props[rep_bins.tolist().index(b)] if b in rep_bins else 0
                          for rep_props, rep_bins in zip(replicate_props, replicate_bins)])
        mean_props.append(props.mean())
        lower_ci.append(np.percentile(props, 5))
        upper_ci.append(np.percentile(props, 95))

    return np.array(all_bins), np.array(mean_props), np.array(lower_ci), np.array(upper_ci)